In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt

In [2]:
TARGET_PATH = 'marine.csv'

In [3]:
df = pd.read_csv(TARGET_PATH, encoding='euc-kr')

In [4]:
df.shape

(929, 8)

In [5]:
df.head()

,항구청명,사용년월,사용월,시설코드,시설서브코드,시설명,접안시간,처리실적
0,감천,2025,8,MKC,03,감천 중앙부두 3선석,401,66795
1,감천,2025,8,MK5,01,감천 5부두 1선석,902,15
2,감천,2025,8,MK4,01,감천 4부두 1선석(국제수산물부두),489,604
3,감천,2025,8,MK1,02,감천 1부두 2선석,1103,2350
4,감천,2025,8,MK2,03,감천 2부두 3선석,233,8064


In [6]:
df.columns.to_list()

['항구청명', '사용년월', '사용월', '시설코드', '시설서브코드', '시설명', '접안시간', '처리실적']

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 929 entries, 0 to 928
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   항구청명    929 non-null    str  
 1   사용년월    929 non-null    int64
 2   사용월     929 non-null    int64
 3   시설코드    929 non-null    str  
 4   시설서브코드  929 non-null    str  
 5   시설명     929 non-null    str  
 6   접안시간    929 non-null    int64
 7   처리실적    929 non-null    int64
dtypes: int64(4), str(4)
memory usage: 92.0 KB


In [8]:
df.describe

<bound method NDFrame.describe of      항구청명  사용년월  사용월 시설코드 시설서브코드                  시설명  접안시간     처리실적
0      감천  2025    8  MKC     03          감천 중앙부두 3선석   401    66795
1      감천  2025    8  MK5     01           감천 5부두 1선석   902       15
2      감천  2025    8  MK4     01  감천 4부두 1선석(국제수산물부두)   489      604
3      감천  2025    8  MK1     02           감천 1부두 2선석  1103     2350
4      감천  2025    8  MK2     03           감천 2부두 3선석   233     8064
..    ...   ...  ...  ...    ...                  ...   ...      ...
924  부산신항  2026    7  MSN     05           신항 2부두 2선석   686  2037022
925  부산신항  2026    7  MS3     03           신항 3부두 3선석   665  5685081
926  부산신항  2026    7  MS5     01           신항 5부두 1선석   594  2860491
927  부산신항  2026    7  MS6     02           신항 6부두 2선석   631  1604299
928  부산신항  2026    7  MS6     01           신항 6부두 1선석   706  1686105

[929 rows x 8 columns]>

### KPI
- 각 항구 별 월간 총 처리실적 : 항만 전체의 핵심 성과와 물동량 추세 측정
- 접안시간당 처리실적 평균 : 전체 평균을 구해서 각 항구 별 평균을 비교해서 측정 할 예정
- 저생산성 선석 비율 : 개선이 필요한 시설의 규모 파악

In [9]:
df.loc[df['시설명'].str.startswith("감천", na=False), "권역"] = "감천"

In [10]:
df.loc[df["시설명"].str.startswith("신항", na=False), "권역"] = "신항"

In [11]:
df.loc[~df["권역"].isin(["감천", "신항"]),"권역"] = "북항"

In [12]:
df.groupby(["권역", "시설명"])

df.groupby

<bound method DataFrame.groupby of      항구청명  사용년월  사용월 시설코드 시설서브코드                  시설명  접안시간     처리실적  권역
0      감천  2025    8  MKC     03          감천 중앙부두 3선석   401    66795  감천
1      감천  2025    8  MK5     01           감천 5부두 1선석   902       15  감천
2      감천  2025    8  MK4     01  감천 4부두 1선석(국제수산물부두)   489      604  감천
3      감천  2025    8  MK1     02           감천 1부두 2선석  1103     2350  감천
4      감천  2025    8  MK2     03           감천 2부두 3선석   233     8064  감천
..    ...   ...  ...  ...    ...                  ...   ...      ...  ..
924  부산신항  2026    7  MSN     05           신항 2부두 2선석   686  2037022  신항
925  부산신항  2026    7  MS3     03           신항 3부두 3선석   665  5685081  신항
926  부산신항  2026    7  MS5     01           신항 5부두 1선석   594  2860491  신항
927  부산신항  2026    7  MS6     02           신항 6부두 2선석   631  1604299  신항
928  부산신항  2026    7  MS6     01           신항 6부두 1선석   706  1686105  신항

[929 rows x 9 columns]>

In [13]:
RRC = (
    df.groupby("권역")
    .size()
    .reset_index(name="데이터개수")
)

print(RRC)

   권역  데이터개수
0  감천    297
1  북항    301
2  신항    331


In [14]:
df['연월'] = (
    df['사용년월'].astype(str).str.strip()
    + '-'
    + df['사용월'].astype(str).str.strip().str.zfill(2)
)

In [15]:
MT = (
    df.groupby(['연월', '권역'])['처리실적']
    .sum()
    .reset_index(name='월간총처리실적')
)

MT #월간총처리실적

,연월,권역,월간총처리실적
0,2025-08,감천,429819
1,2025-08,북항,8767414
2,2025-08,신항,26753088
3,2025-09,감천,427012
4,2025-09,북항,8423832
5,2025-09,신항,27723116
6,2025-10,감천,388344
7,2025-10,북항,8378595
8,2025-10,신항,27349609
9,2025-11,감천,411874


In [16]:
result = (
        df.groupby(["연월", "권역"], as_index=False)
        .agg(
            총접안시간=("접안시간", "sum"),
            총처리실적=("처리실적", "sum"),
        )
    )

result

,연월,권역,총접안시간,총처리실적
0,2025-08,감천,14418,429819
1,2025-08,북항,13267,8767414
2,2025-08,신항,13140,26753088
3,2025-09,감천,11743,427012
4,2025-09,북항,11808,8423832
5,2025-09,신항,13881,27723116
6,2025-10,감천,10104,388344
7,2025-10,북항,12217,8378595
8,2025-10,신항,13523,27349609
9,2025-11,감천,10975,411874


In [17]:
result['시간당처리실적'] = (result['총처리실적'] / result['총접안시간']).round(2)

result['시간당처리실적']

0       29.81
1      660.84
2     2036.00
3       36.36
4      713.40
5     1997.20
6       38.43
7      685.81
8     2022.45
9       37.53
10     788.30
11    2107.68
12      37.34
13     732.76
14    2097.22
15      32.91
16     742.51
17    2129.33
18      25.47
19     704.56
20    2222.32
21      23.94
22     748.13
23    2130.80
24      28.22
25     792.77
26    2093.06
27      18.67
28     758.12
29    2151.55
30      27.07
31     744.40
32    2153.62
33      27.10
34     738.96
35    2352.13
Name: 시간당처리실적, dtype: float64

In [18]:
RWA = (result.groupby("권역", as_index=False).agg(전체접안시간=("총접안시간", "sum"),전체처리실적=("총처리실적", "sum")))

RWA


,권역,전체접안시간,전체처리실적
0,감천,154028,4495703
1,북항,143395,105160469
2,신항,166973,355304748


In [19]:
RWA['평균접안시간당처리실적'] = (RWA['전체처리실적'] / RWA['전체접안시간']).round(2)

RWA

,권역,전체접안시간,전체처리실적,평균접안시간당처리실적
0,감천,154028,4495703,29.19
1,북항,143395,105160469,733.36
2,신항,166973,355304748,2127.92


In [20]:
#평균접안시간당처리실적을 시간당처리실적과 비교해서 몇%가 평균 미만의 처리실적을 가지고 있는지 분석하기

comparison = result.merge(
    RWA[['권역','평균접안시간당처리실적']],
    on="권역",
    how="left",
    validate="many_to_one"
)

comparison

,연월,권역,총접안시간,총처리실적,시간당처리실적,평균접안시간당처리실적
0,2025-08,감천,14418,429819,29.81,29.19
1,2025-08,북항,13267,8767414,660.84,733.36
2,2025-08,신항,13140,26753088,2036.00,2127.92
3,2025-09,감천,11743,427012,36.36,29.19
4,2025-09,북항,11808,8423832,713.40,733.36
5,2025-09,신항,13881,27723116,1997.20,2127.92
6,2025-10,감천,10104,388344,38.43,29.19
7,2025-10,북항,12217,8378595,685.81,733.36
8,2025-10,신항,13523,27349609,2022.45,2127.92
9,2025-11,감천,10975,411874,37.53,29.19


In [21]:
comparison['평균대비차이율'] = (
    (comparison['시간당처리실적'] - comparison['평균접안시간당처리실적'])
    / comparison['평균접안시간당처리실적'] * 100).round(2)

comparison['평균대비차이율']

0      2.12
1     -9.89
2     -4.32
3     24.56
4     -2.72
5     -6.14
6     31.65
7     -6.48
8     -4.96
9     28.57
10     7.49
11    -0.95
12    27.92
13    -0.08
14    -1.44
15    12.74
16     1.25
17     0.07
18   -12.74
19    -3.93
20     4.44
21   -17.99
22     2.01
23     0.14
24    -3.32
25     8.10
26    -1.64
27   -36.04
28     3.38
29     1.11
30    -7.26
31     1.51
32     1.21
33    -7.16
34     0.76
35    10.54
Name: 평균대비차이율, dtype: float64

In [22]:
comparison.sort_values('권역')

,연월,권역,총접안시간,총처리실적,시간당처리실적,평균접안시간당처리실적,평균대비차이율
0,2025-08,감천,14418,429819,29.81,29.19,2.12
3,2025-09,감천,11743,427012,36.36,29.19,24.56
6,2025-10,감천,10104,388344,38.43,29.19,31.65
9,2025-11,감천,10975,411874,37.53,29.19,28.57
12,2025-12,감천,9664,360885,37.34,29.19,27.92
15,2026-01,감천,9195,302586,32.91,29.19,12.74
18,2026-02,감천,12675,322851,25.47,29.19,-12.74
21,2026-03,감천,14931,357519,23.94,29.19,-17.99
24,2026-04,감천,15144,427321,28.22,29.19,-3.32
27,2026-05,감천,18539,346067,18.67,29.19,-36.04


In [23]:
comparison["평균초과여부"] = (comparison["평균대비차이율"] > 0)

comparison['평균초과여부']

0      True
1     False
2     False
3      True
4     False
5     False
6      True
7     False
8     False
9      True
10     True
11    False
12     True
13    False
14    False
15     True
16     True
17     True
18    False
19    False
20     True
21    False
22     True
23     True
24    False
25     True
26    False
27    False
28     True
29     True
30    False
31     True
32     True
33    False
34     True
35     True
Name: 평균초과여부, dtype: bool